# Estimasi Model LSTM — Prediksi Harga Batu Bara Newcastle
Skripsi: *Perbandingan Prediksi Harga Batu Bara Newcastle dengan Markov Switching Autoregressive (MS-AR) dan Long Short-Term Memory (LSTM)*

Notebook ini mengikuti prosedur Bab III:
- Split data 90% training : 10% testing
- Normalisasi Min-Max ke rentang [0.1, 1] (di-fit dari data training saja)
- Sliding window (time steps) untuk membentuk data supervised learning
- Grid search 72 kombinasi hyperparameter (time steps, units, batch size, learning rate)
- Early stopping (patience = 100, epoch maksimum = 1000)
- Prediksi data testing secara dynamic/recursive multi-step
- Evaluasi RMSE, MAE, MAPE


## 1. Install & Import Library

In [ ]:
!pip install -q tensorflow scikit-learn pandas numpy matplotlib openpyxl

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
import itertools
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

## 2. Upload & Load Data
Upload file `.csv` atau `.xlsx` dengan kolom `Date` dan `Price` (data harga batu bara Newcastle mingguan).

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload file csv/xlsx harga mingguan (kolom: Date, Price)

filename = list(uploaded.keys())[0]
if filename.endswith('.xlsx'):
    df = pd.read_excel(filename)
else:
    df = pd.read_csv(filename)

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
prices = df['Price'].values.reshape(-1, 1)
print("Jumlah observasi:", len(prices))
df.tail()

## 3. Split Data 90:10 (Training : Testing)

In [ ]:
n = len(prices)
train_size = int(np.ceil(0.9 * n))   # 90% training
train_data = prices[:train_size]
test_data  = prices[train_size:]

print(f"Training: {len(train_data)} observasi")
print(f"Testing : {len(test_data)} observasi")

## 4. Normalisasi Min-Max [0,1 – 1]
Scaler di-*fit* hanya dari data training untuk menghindari *data leakage*.

In [ ]:
scaler = MinMaxScaler(feature_range=(0.1, 1))
scaler.fit(train_data)

train_scaled = scaler.transform(train_data)
test_scaled  = scaler.transform(test_data)

print("Min training asli:", scaler.data_min_[0], "| Max training asli:", scaler.data_max_[0])

## 5. Fungsi Sliding Window (Pembentukan Sekuens)

In [ ]:
def create_sequences(data, time_steps):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i + time_steps, 0])
        y.append(data[i + time_steps, 0])
    return np.array(X), np.array(y)

## 6. Fungsi Bangun Arsitektur Model LSTM
1 hidden layer LSTM + 1 Dense output layer, optimizer SGD, loss MSE.

In [ ]:
def build_model(time_steps, units, learning_rate):
    model = Sequential([
        LSTM(units, activation='tanh', input_shape=(time_steps, 1)),
        Dense(1)
    ])
    model.compile(optimizer=SGD(learning_rate=learning_rate), loss='mse')
    return model

## 7. Grid Search (72 Kombinasi Hyperparameter)
time steps × units × batch size × learning rate = 3 × 3 × 4 × 2 = 72 kombinasi.

**Catatan:** proses ini bisa lama — aktifkan GPU (`Runtime > Change runtime type > GPU`).

In [ ]:
time_steps_list    = [5, 10, 20]
units_list         = [16, 32, 64]
batch_size_list    = [8, 16, 32, 64]
learning_rate_list = [0.01, 0.001]

max_epoch = 1000
patience  = 100

results = []
combo_id = 0
total_combo = len(time_steps_list) * len(units_list) * len(batch_size_list) * len(learning_rate_list)

for time_steps, units, batch_size, lr in itertools.product(
        time_steps_list, units_list, batch_size_list, learning_rate_list):

    combo_id += 1
    print(f"\n[{combo_id}/{total_combo}] time_steps={time_steps}, units={units}, "
          f"batch_size={batch_size}, lr={lr}")

    # Bentuk sekuens dari data training (yang sudah dinormalisasi)
    X_train_full, y_train_full = create_sequences(train_scaled, time_steps)

    # Split train_full -> train_sub (85%) & validation (15%), tetap berurutan waktu
    val_size = int(len(X_train_full) * 0.15)
    X_train_sub, y_train_sub = X_train_full[:-val_size], y_train_full[:-val_size]
    X_val, y_val             = X_train_full[-val_size:], y_train_full[-val_size:]

    X_train_sub = X_train_sub.reshape((X_train_sub.shape[0], time_steps, 1))
    X_val       = X_val.reshape((X_val.shape[0], time_steps, 1))

    model = build_model(time_steps, units, lr)

    es = EarlyStopping(monitor='val_loss', patience=patience,
                        restore_best_weights=True, verbose=0)

    history = model.fit(
        X_train_sub, y_train_sub,
        validation_data=(X_val, y_val),
        epochs=max_epoch,
        batch_size=batch_size,
        callbacks=[es],
        verbose=0
    )

    best_val_loss = min(history.history['val_loss'])
    best_epoch = np.argmin(history.history['val_loss']) + 1

    results.append({
        'time_steps': time_steps,
        'units': units,
        'batch_size': batch_size,
        'learning_rate': lr,
        'val_loss': best_val_loss,
        'best_epoch': best_epoch
    })

    print(f"  -> val_loss terbaik: {best_val_loss:.6f} (epoch {best_epoch})")

results_df = pd.DataFrame(results).sort_values('val_loss').reset_index(drop=True)
results_df.to_csv('grid_search_results.csv', index=False)
print("\n=== 5 Kombinasi Terbaik ===")
print(results_df.head())

## 8. Ambil Hyperparameter Terbaik & Latih Ulang Model Final

In [ ]:
best_params = results_df.iloc[0]
time_steps  = int(best_params['time_steps'])
units       = int(best_params['units'])
batch_size  = int(best_params['batch_size'])
lr          = best_params['learning_rate']

print("Hyperparameter terbaik:")
print(best_params)

# Bentuk ulang sekuens dengan time_steps terbaik
X_train_full, y_train_full = create_sequences(train_scaled, time_steps)
val_size = int(len(X_train_full) * 0.15)
X_train_sub, y_train_sub = X_train_full[:-val_size], y_train_full[:-val_size]
X_val, y_val             = X_train_full[-val_size:], y_train_full[-val_size:]

X_train_sub = X_train_sub.reshape((X_train_sub.shape[0], time_steps, 1))
X_val       = X_val.reshape((X_val.shape[0], time_steps, 1))

final_model = build_model(time_steps, units, lr)
es = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True, verbose=1)

final_history = final_model.fit(
    X_train_sub, y_train_sub,
    validation_data=(X_val, y_val),
    epochs=max_epoch,
    batch_size=batch_size,
    callbacks=[es],
    verbose=1
)

## 9. Prediksi Data Testing (Dynamic/Recursive Multi-Step Forecast)

In [ ]:
# Ambil time_steps terakhir dari data TRAINING (scaled) sebagai titik awal rekursif
last_window = train_scaled[-time_steps:].flatten().tolist()

predictions_scaled = []
n_test = len(test_data)

for _ in range(n_test):
    x_input = np.array(last_window[-time_steps:]).reshape((1, time_steps, 1))
    pred = final_model.predict(x_input, verbose=0)[0, 0]
    predictions_scaled.append(pred)
    last_window.append(pred)   # dorong prediksi masuk ke window (dynamic/recursive)

predictions_scaled = np.array(predictions_scaled).reshape(-1, 1)

# Denormalisasi kembali ke skala harga aktual (USD/ton)
predictions = scaler.inverse_transform(predictions_scaled)
actual = test_data

print("Prediksi vs Aktual:")
for i in range(n_test):
    print(f"Minggu {i+1}: Aktual={actual[i,0]:.2f} | Prediksi={predictions[i,0]:.2f}")

## 10. Evaluasi Akurasi (RMSE, MAE, MAPE)

In [ ]:
rmse = np.sqrt(np.mean((actual - predictions) ** 2))
mae  = np.mean(np.abs(actual - predictions))
mape = np.mean(np.abs((actual - predictions) / actual)) * 100

print(f"RMSE = {rmse:.4f}")
print(f"MAE  = {mae:.4f}")
print(f"MAPE = {mape:.4f}%")

## 11. Plot Perbandingan Aktual vs Prediksi

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, n_test+1), actual, marker='o', label='Aktual')
plt.plot(range(1, n_test+1), predictions, marker='s', linestyle='--', label='Prediksi LSTM')
plt.xlabel('Minggu ke-')
plt.ylabel('Harga Batu Bara Newcastle (USD/ton)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('prediksi_lstm_testing.png', dpi=200)
plt.show()